### Setup

In [ ]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
import json
from diffusers import StableDiffusionPipeline

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 100

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
    
## Data
ANSWER_INSTRUCTION_FOLDER = config.get("ANSWER_INSTRUCTION_FOLDER")
ANSWER_INSTRUCTION_FILE = sorted(os.listdir(ANSWER_INSTRUCTION_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = config.get("SD_IMAGE_FOLDER")
FIGMA_IMAGE_FOLDER = config.get("FIGMA_IMAGE_FOLDER")

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = ''

### Generate Image by Stable Diffusion

In [ ]:
# :oad Model
satblediffusion_pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16).to(device);
# Generate Images
for idx in range(len(ANSWER_INSTRUCTION_FILE)):
    instrucion_path = os.path.join(ANSWER_INSTRUCTION_FOLDER, ANSWER_INSTRUCTION_FILE[idx])
    with open(instrucion_path, 'r') as f:
        instruction = f.read()[16:]
    generated_image = satblediffusion_pipe(instruction).images[0]
    save_image_path = SD_IMAGE_FOLDER + "/" + ANSWER_INSTRUCTION_FILE[idx][:-5] + ".jpg"
    generated_image.save(save_image_path)

### Generate Image by FIGMA

In [ ]:
# Load Saved Model
figma_pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16).to(device);
figma_pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True)

# Generate Images
for idx in range(len(ANSWER_INSTRUCTION_FILE)):
    instrucion_path = os.path.join(ANSWER_INSTRUCTION_FOLDER, ANSWER_INSTRUCTION_FILE[idx])
    with open(instrucion_path, 'r') as f:
        instruction = f.read()[16:]
    generated_image = figma_pipe(instruction).images[0]
    save_image_path = FIGMA_IMAGE_FOLDER + "/" + ANSWER_INSTRUCTION_FILE[idx][:-5] + ".jpg"
    generated_image.save(save_image_path)